# Modèle de Machine Learning pour la Prédiction des Catégories d'Obésité

Ce notebook développe un modèle multiclasse pour prédire les catégories d'obésité basé sur des données de santé et de mode de vie.

## Objectif
Automatiser l'évaluation du risque d'obésité en utilisant des modèles de Machine Learning pour aider les organisations de santé.


In [ ]:
# Importation des bibliothèques nécessaires
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import precision_recall_fscore_support
import joblib
import warnings
warnings.filterwarnings('ignore')

# Configuration pour l'affichage
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)


In [ ]:
# Chargement et exploration des données
df = pd.read_csv('Data.csv')

print("=== INFORMATIONS SUR LE DATASET ===")
print(f"Forme du dataset: {df.shape}")
print(f"Nombre de lignes: {df.shape[0]}")
print(f"Nombre de colonnes: {df.shape[1]}")

print("\n=== INFORMATIONS SUR LES COLONNES ===")
print(df.info())

print("\n=== PREMIÈRES LIGNES ===")
print(df.head())

print("\n=== STATISTIQUES DESCRIPIVES ===")
print(df.describe())


In [ ]:
# Analyse de la variable cible (NObeyesdad)
print("=== DISTRIBUTION DES CATÉGORIES D'OBÉSITÉ ===")
obesity_counts = df['NObeyesdad'].value_counts()
print(obesity_counts)

# Visualisation de la distribution
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
obesity_counts.plot(kind='bar', color='skyblue')
plt.title('Distribution des Catégories d\'Obésité')
plt.xlabel('Catégorie d\'Obésité')
plt.ylabel('Nombre d\'individus')
plt.xticks(rotation=45, ha='right')

plt.subplot(1, 2, 2)
plt.pie(obesity_counts.values, labels=obesity_counts.index, autopct='%1.1f%%', startangle=90)
plt.title('Répartition en Pourcentage')

plt.tight_layout()
plt.show()

print(f"\nNombre de catégories uniques: {df['NObeyesdad'].nunique()}")
print(f"Catégories: {list(df['NObeyesdad'].unique())}")


In [ ]:
# Analyse des variables catégorielles
categorical_columns = df.select_dtypes(include=['object']).columns.tolist()
print("=== VARIABLES CATÉGORIELLES ===")
print(f"Colonnes catégorielles: {categorical_columns}")

# Affichage des valeurs uniques pour chaque variable catégorielle
for col in categorical_columns:
    print(f"\n{col}:")
    print(f"  Valeurs uniques: {df[col].unique()}")
    print(f"  Nombre de valeurs uniques: {df[col].nunique()}")

# Vérification des valeurs manquantes
print("\n=== VALEURS MANQUANTES ===")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])


In [ ]:
# Préprocessing des données
print("=== PRÉPROCESSING DES DONNÉES ===")

# Création d'une copie pour le preprocessing
df_processed = df.copy()

# Encodage des variables catégorielles
label_encoders = {}
categorical_features = ['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']

for feature in categorical_features:
    le = LabelEncoder()
    df_processed[feature] = le.fit_transform(df_processed[feature])
    label_encoders[feature] = le
    print(f"Encodé {feature}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Encodage de la variable cible
target_encoder = LabelEncoder()
df_processed['NObeyesdad_encoded'] = target_encoder.fit_transform(df_processed['NObeyesdad'])
print(f"\nEncodé NObeyesdad: {dict(zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_)))}")

print(f"\nForme après preprocessing: {df_processed.shape}")
print("\nPremières lignes après preprocessing:")
print(df_processed.head())


In [ ]:
# Préparation des features et de la variable cible
# Sélection des features (exclure la variable cible originale)
feature_columns = [col for col in df_processed.columns if col not in ['NObeyesdad', 'NObeyesdad_encoded']]
X = df_processed[feature_columns]
y = df_processed['NObeyesdad_encoded']

print("=== PRÉPARATION DES DONNÉES ===")
print(f"Features sélectionnées: {feature_columns}")
print(f"Forme de X: {X.shape}")
print(f"Forme de y: {y.shape}")

# Division train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nDivision train/test:")
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

# Normalisation des features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nAprès normalisation:")
print(f"X_train_scaled: {X_train_scaled.shape}")
print(f"X_test_scaled: {X_test_scaled.shape}")


In [ ]:
# Entraînement de plusieurs modèles
print("=== ENTRAÎNEMENT DES MODÈLES ===")

models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'SVM': SVC(random_state=42, probability=True)
}

model_results = {}

for name, model in models.items():
    print(f"\nEntraînement de {name}...")
    
    # Entraînement
    if name == 'Logistic Regression' or name == 'SVM':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_pred_proba = model.predict_proba(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)
    
    # Évaluation
    accuracy = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')
    
    model_results[name] = {
        'model': model,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'predictions': y_pred,
        'probabilities': y_pred_proba
    }
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")


In [ ]:
# Comparaison des performances des modèles
print("=== COMPARAISON DES PERFORMANCES ===")

# Création d'un DataFrame pour la comparaison
comparison_df = pd.DataFrame({
    'Modèle': list(model_results.keys()),
    'Accuracy': [model_results[name]['accuracy'] for name in model_results.keys()],
    'Precision': [model_results[name]['precision'] for name in model_results.keys()],
    'Recall': [model_results[name]['recall'] for name in model_results.keys()],
    'F1-Score': [model_results[name]['f1_score'] for name in model_results.keys()]
})

print(comparison_df.round(4))

# Visualisation des performances
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

for i, metric in enumerate(metrics):
    ax = axes[i//2, i%2]
    bars = ax.bar(comparison_df['Modèle'], comparison_df[metric], color='skyblue', alpha=0.7)
    ax.set_title(f'Comparaison des {metric}')
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1)
    
    # Ajout des valeurs sur les barres
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.3f}', ha='center', va='bottom')
    
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

# Sélection du meilleur modèle
best_model_name = comparison_df.loc[comparison_df['F1-Score'].idxmax(), 'Modèle']
best_model = model_results[best_model_name]['model']
print(f"\nMeilleur modèle: {best_model_name}")
print(f"F1-Score: {comparison_df.loc[comparison_df['F1-Score'].idxmax(), 'F1-Score']:.4f}")


In [ ]:
# Analyse détaillée du meilleur modèle
print(f"=== ANALYSE DÉTAILLÉE DU MEILLEUR MODÈLE: {best_model_name} ===")

# Rapport de classification détaillé
y_pred_best = model_results[best_model_name]['predictions']
print("\nRapport de Classification:")
print(classification_report(y_test, y_pred_best, target_names=target_encoder.classes_))

# Matrice de confusion
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_encoder.classes_, 
            yticklabels=target_encoder.classes_)
plt.title(f'Matrice de Confusion - {best_model_name}')
plt.xlabel('Prédictions')
plt.ylabel('Valeurs Réelles')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Importance des features (si Random Forest ou Gradient Boosting)
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': feature_columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    plt.figure(figsize=(12, 8))
    sns.barplot(data=feature_importance.head(10), x='importance', y='feature')
    plt.title(f'Top 10 des Features les Plus Importantes - {best_model_name}')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()
    
    print("\nTop 10 des Features les Plus Importantes:")
    print(feature_importance.head(10))


In [ ]:
# Sauvegarde du modèle et des préprocesseurs
print("=== SAUVEGARDE DU MODÈLE ===")

# Création du dossier models s'il n'existe pas
import os
os.makedirs('models', exist_ok=True)

# Sauvegarde du meilleur modèle
joblib.dump(best_model, 'models/best_obesity_model.pkl')
print(f"Modèle {best_model_name} sauvegardé dans models/best_obesity_model.pkl")

# Sauvegarde du scaler
joblib.dump(scaler, 'models/scaler.pkl')
print("Scaler sauvegardé dans models/scaler.pkl")

# Sauvegarde des encodeurs
joblib.dump(label_encoders, 'models/label_encoders.pkl')
print("Label encoders sauvegardés dans models/label_encoders.pkl")

# Sauvegarde de l'encodeur de la variable cible
joblib.dump(target_encoder, 'models/target_encoder.pkl')
print("Target encoder sauvegardé dans models/target_encoder.pkl")

# Sauvegarde des noms des features
joblib.dump(feature_columns, 'models/feature_columns.pkl')
print("Feature columns sauvegardées dans models/feature_columns.pkl")

print("\nTous les fichiers ont été sauvegardés avec succès!")
